# House Edge
Slot Machine Forecasting / TAMU Datathon 2026

Use the play history to put odds on the next spin. Run all cells with Python 3.11+ to compare six models and create a submission. Your data is in public/. Start with README.md, then check RULES.md before submitting.


In [ ]:
import os
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError('Select a Python 3.11 or newer notebook kernel.')
ROOT = Path.cwd()
if not (ROOT / 'slot_forecasting').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'slot_forecasting').is_dir():
    raise RuntimeError('Open this notebook from the repository or its notebooks folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from slot_forecasting.starter import public_directory, run_starter
from slot_forecasting.baseline import FrequencyForecaster
from slot_forecasting.io import load_inputs, read_submission
from slot_forecasting.scoring import mean_payout, payout_quantile

DATA = ROOT / 'public'
OUTPUT = ROOT / 'outputs'
PUBLIC = public_directory(DATA)
print('Public data:', PUBLIC)


## Start with the pooled model
The pooled starter learns payout frequencies from machines with the same type and signal. It combines those frequencies with the current machine's history. `prior_strength=20` gives the shared frequencies the weight of 20 observations, which helps with short histories and rare payouts.

The next cells compare all six baselines on validation data, then export the fixed pooled model. To try another model from the terminal, use `--method`. Choose your settings on validation data and keep the final test answers out of training.


In [ ]:
report = run_starter(PUBLIC, OUTPUT)
for method, result in report['comparison'].items():
    if 'contest' in result:
        print(method, f"technical points / {result['contest']['max_points']:g}:", round(result['contest']['technical_points'], 6))
    else:
        print(method, 'legacy diagnostic / 100:', round(result['score'], 6))
print('Best validation method:', report['best_validation_method'])
print('Exported settings:', report['settings'])
print('Total workflow seconds:', round(report['runtime_seconds']['total'], 3))


## Read the odds
The five probabilities describe your forecast. Their weighted mean is the expected gross payout. The 10th and 90th payout quantiles give a central 80% range; discrete outcomes can make its coverage exceed 80%.

Main is worth 60 points and adaptation is worth 40. Within each round, divide each condition's squared probability error by its frozen starter error (floor `1e-9`), then average those four ratios into `R`. Points are `round weight / (1 + R)`. The pooled reference scores 30 in main and 20 in adaptation.

When the adaptation kit opens, run your workflow on its new training and validation data. Use a new output folder so you keep the main run. Check the released rules for the scoring metric if you are using an older practice dataset.


In [ ]:
validation_queries, _ = load_inputs(PUBLIC / 'validation')
predictions = read_submission(OUTPUT / 'validation_predictions.csv', {q.query_id for q in validation_queries})
query_id = validation_queries[0].query_id
p = predictions[query_id]
print('Query:', query_id)
print('Payout probabilities:', dict(zip((0, 1, 2, 5, 20), p)))
print('Expected gross payout:', round(mean_payout(p), 3))
print('Central 80% payout interval:', (payout_quantile(p, .1), payout_quantile(p, .9)))


## Keep a reproducible submission
The workflow saved `baseline.json`, loaded it again, and checked the predictions against the test query IDs. It also checked that the saved model reproduces validation probabilities within a tolerance of `1e-14`.

Keep `run.json`: it records your environment, settings, scores, timings, and file hashes. Unchanged code and inputs recreate the same predictions in the same Python environment; timings may vary. Rerunning the notebook replaces its output files, so use a new `SLOTS_OUTPUT_DIR` to keep a run.


In [ ]:
model = FrequencyForecaster.load(OUTPUT / 'baseline.json')
test_queries, test_history = load_inputs(PUBLIC / 'test')
submission_csv = OUTPUT / 'submission.csv'
test_predictions = read_submission(submission_csv, {q.query_id for q in test_queries})
reproduced = model.predict(test_queries, test_history)
assert all(abs(p - q) <= 1e-14 for key in test_predictions
           for p, q in zip(test_predictions[key], reproduced[key]))
print('Valid test forecasts:', len(test_predictions))
print('Submit:', submission_csv.resolve())
print('Run record:', (OUTPUT / 'run.json').resolve())
print('Checks:', report['checks'])


## Find your edge
In an extracted kit, run `python run_experiments.py --out experiments-01`. It compares 12 settings, selects a model on tuning episodes, then checks that choice on separate audit episodes. See `EXPERIMENTS.md` for how to add a candidate.

Try a pair of earlier spins, a shorter history window, or a separate model for each machine. Compare condition scores before keeping a change.


In [ ]:
for name, result in report['comparison'].items():
    print(name, 'validation technical points:', round(result['contest']['technical_points'], 3))
